In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# 1. Khai báo đường dẫn
raw_dir = '../data/raw/meta/'
processed_dir = '../data/processed/'
os.makedirs(processed_dir, exist_ok=True)

try:
    # 2. Đọc file meta
    df = pd.read_csv(os.path.join(raw_dir, 'meta.csv'))
    print(f"Tổng số ca bệnh: {len(df)}")
    
    # Điền giá trị thiếu
    df['sex'] = df['sex'].fillna('Unknown')
    df['location'] = df['location'].fillna('Unknown')
    df['elevation'] = df['elevation'].fillna('Unknown')

    # 3. THỰC HIỆN CHIA DỮ LIỆU TỶ LỆ: 60% - 15% - 10% - 15%
    # Bước 3.1: Cắt 15% làm tập Test, còn lại 85%
    rest_df, test_df = train_test_split(df, test_size=0.15, random_state=42)
    
    # Bước 3.2: Từ 85% còn lại, cắt 10% (so với tổng) làm tập Calibration (Tỷ lệ: 10/85)
    rest_df, cal_df = train_test_split(rest_df, test_size=(10/85), random_state=42)
    
    # Bước 3.3: Từ 75% còn lại, cắt 15% làm tập Validation, 60% làm tập Train (Tỷ lệ: 15/75 = 0.2)
    train_df, val_df = train_test_split(rest_df, test_size=(15/75), random_state=42)

    # 4. Kiểm tra số lượng và tính rò rỉ (Leakage Check)
    train_cases = set(train_df['case_num'])
    val_cases = set(val_df['case_num'])
    cal_cases = set(cal_df['case_num'])
    test_cases = set(test_df['case_num'])
    
    # Kiểm tra giao thoa giữa 4 tập
    assert train_cases.isdisjoint(val_cases), "LỖI: Rò rỉ giữa Train và Val!"
    assert train_cases.isdisjoint(cal_cases), "LỖI: Rò rỉ giữa Train và Cal!"
    assert train_cases.isdisjoint(test_cases), "LỖI: Rò rỉ giữa Train và Test!"
    
    print("-" * 40)
    print(f"Tập Huấn luyện (Train - 60%):      {len(train_df)} ca")
    print(f"Tập Xác thực (Validation - 15%):  {len(val_df)} ca")
    print(f"Tập Hiệu chỉnh (Calibration - 10%): {len(cal_df)} ca")
    print(f"Tập Kiểm thử (Test - 15%):        {len(test_df)} ca")
    print("-" * 40)
    print("TUYỆT ĐỐI AN TOÀN: 0% Rò rỉ dữ liệu giữa các tập!")

    # 5. Lưu 4 file ra thư mục processed
    train_df.to_csv(os.path.join(processed_dir, 'train_split.csv'), index=False)
    val_df.to_csv(os.path.join(processed_dir, 'val_split.csv'), index=False)
    cal_df.to_csv(os.path.join(processed_dir, 'cal_split.csv'), index=False)
    test_df.to_csv(os.path.join(processed_dir, 'test_split.csv'), index=False)
    
    print(f"\nĐã lưu 4 file phân chia dữ liệu vào: {processed_dir}")

except Exception as e:
    print(f"Có lỗi xảy ra: {e}")

Tổng số ca bệnh: 1011
----------------------------------------
Tập Huấn luyện (Train - 60%):      605 ca
Tập Xác thực (Validation - 15%):  152 ca
Tập Hiệu chỉnh (Calibration - 10%): 102 ca
Tập Kiểm thử (Test - 15%):        152 ca
----------------------------------------
TUYỆT ĐỐI AN TOÀN: 0% Rò rỉ dữ liệu giữa các tập!

Đã lưu 4 file phân chia dữ liệu vào: ../data/processed/
